In [ ]:
pwd

In [ ]:
import math

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from tqdm.auto import tqdm

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'
# the next line provides graphs of better quality on HiDPI screens
%config InlineBackend.figure_format = 'retina'

plt.style.use('seaborn')

In [ ]:
# this is to use progress_apply, read more at https://pypi.org/project/tqdm/#pandas-integration
tqdm.pandas()

In [ ]:
df_reviews = pd.read_csv('/datasets/imdb_reviews.tsv', sep='\t', dtype={'votes': 'Int64'})

In [ ]:
df_reviews.head(10)

In [ ]:
df_reviews.shape

In [ ]:
df_reviews.info()

In [ ]:
df_reviews.describe()

In [ ]:
df_reviews.isna().sum()


In [ ]:
df_reviews.duplicated().sum()

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(16, 8))

ax = axs[0]

dft1 = df_reviews[['tconst', 'start_year']].drop_duplicates() \
    ['start_year'].value_counts().sort_index()
dft1 = dft1.reindex(index=np.arange(dft1.index.min(), max(dft1.index.max(), 2021))).fillna(0)
dft1.plot(kind='bar', ax=ax)
ax.set_title('Number of Movies Over Years')

ax = axs[1]

dft2 = df_reviews.groupby(['start_year', 'pos'])['pos'].count().unstack()
dft2 = dft2.reindex(index=np.arange(dft2.index.min(), max(dft2.index.max(), 2021))).fillna(0)

dft2.plot(kind='bar', stacked=True, label='#reviews (neg, pos)', ax=ax)

dft2 = df_reviews['start_year'].value_counts().sort_index()
dft2 = dft2.reindex(index=np.arange(dft2.index.min(), max(dft2.index.max(), 2021))).fillna(0)
dft3 = (dft2/dft1).fillna(0)
axt = ax.twinx()
dft3.reset_index(drop=True).rolling(5).mean().plot(color='orange', label='reviews per movie (avg over 5 years)', ax=axt)

lines, labels = axt.get_legend_handles_labels()
ax.legend(lines, labels, loc='upper left')

ax.set_title('Number of Reviews Over Years')

fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(16, 5))

ax = axs[0]
dft = df_reviews.groupby('tconst')['review'].count() \
    .value_counts() \
    .sort_index()
dft.plot.bar(ax=ax)
ax.set_title('Bar Plot of #Reviews Per Movie')

ax = axs[1]
dft = df_reviews.groupby('tconst')['review'].count()
sns.kdeplot(dft, ax=ax)
ax.set_title('KDE Plot of #Reviews Per Movie')

fig.tight_layout()

In [ ]:
df_reviews['pos'].value_counts()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

ax = axs[0]
dft = df_reviews.query('ds_part == "train"')['rating'].value_counts().sort_index()
dft = dft.reindex(index=np.arange(min(dft.index.min(), 1), max(dft.index.max(), 11))).fillna(0)
dft.plot.bar(ax=ax)
ax.set_ylim([0, 5000])
ax.set_title('The train set: distribution of ratings')

ax = axs[1]
dft = df_reviews.query('ds_part == "test"')['rating'].value_counts().sort_index()
dft = dft.reindex(index=np.arange(min(dft.index.min(), 1), max(dft.index.max(), 11))).fillna(0)
dft.plot.bar(ax=ax)
ax.set_ylim([0, 5000])
ax.set_title('The test set: distribution of ratings')

fig.tight_layout()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 8), gridspec_kw=dict(width_ratios=(2, 1), height_ratios=(1, 1)))

ax = axs[0][0]

dft = df_reviews.query('ds_part == "train"').groupby(['start_year', 'pos'])['pos'].count().unstack()
dft.index = dft.index.astype('int')
dft = dft.reindex(index=np.arange(dft.index.min(), max(dft.index.max(), 2020))).fillna(0)
dft.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('The train set: number of reviews of different polarities per year')

ax = axs[0][1]

dft = df_reviews.query('ds_part == "train"').groupby(['tconst', 'pos'])['pos'].count().unstack()
sns.kdeplot(dft[0], color='blue', label='negative', kernel='epa', ax=ax)
sns.kdeplot(dft[1], color='green', label='positive', kernel='epa', ax=ax)
ax.legend()
ax.set_title('The train set: distribution of different polarities per movie')

ax = axs[1][0]

dft = df_reviews.query('ds_part == "test"').groupby(['start_year', 'pos'])['pos'].count().unstack()
dft.index = dft.index.astype('int')
dft = dft.reindex(index=np.arange(dft.index.min(), max(dft.index.max(), 2020))).fillna(0)
dft.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('The test set: number of reviews of different polarities per year')

ax = axs[1][1]

dft = df_reviews.query('ds_part == "test"').groupby(['tconst', 'pos'])['pos'].count().unstack()
sns.kdeplot(dft[0], color='blue', label='negative', kernel='epa', ax=ax)
sns.kdeplot(dft[1], color='green', label='positive', kernel='epa', ax=ax)
ax.legend()
ax.set_title('The test set: distribution of different polarities per movie')

fig.tight_layout()

In [ ]:
df_reviews['pos'].value_counts(normalize=True)

In [ ]:
import sklearn.metrics as metrics

def evaluate_model(model, train_features, train_target, test_features, test_target):
    
    eval_stats = {}
    
    fig, axs = plt.subplots(1, 3, figsize=(20, 6)) 
    
    for type, features, target in (('train', train_features, train_target), ('test', test_features, test_target)):
        
        eval_stats[type] = {}
    
        pred_target = model.predict(features)
        pred_proba = model.predict_proba(features)[:, 1]
        
        # F1
        f1_thresholds = np.arange(0, 1.01, 0.05)
        f1_scores = [metrics.f1_score(target, pred_proba>=threshold) for threshold in f1_thresholds]
        
        # ROC
        fpr, tpr, roc_thresholds = metrics.roc_curve(target, pred_proba)
        roc_auc = metrics.roc_auc_score(target, pred_proba)    
        eval_stats[type]['ROC AUC'] = roc_auc

        # PRC
        precision, recall, pr_thresholds = metrics.precision_recall_curve(target, pred_proba)
        aps = metrics.average_precision_score(target, pred_proba)
        eval_stats[type]['APS'] = aps
        
        if type == 'train':
            color = 'blue'
        else:
            color = 'green'

        # F1 Score
        ax = axs[0]
        max_f1_score_idx = np.argmax(f1_scores)
        ax.plot(f1_thresholds, f1_scores, color=color, label=f'{type}, max={f1_scores[max_f1_score_idx]:.2f} @ {f1_thresholds[max_f1_score_idx]:.2f}')
        # setting crosses for some thresholds
        for threshold in (0.2, 0.4, 0.5, 0.6, 0.8):
            closest_value_idx = np.argmin(np.abs(f1_thresholds-threshold))
            marker_color = 'orange' if threshold != 0.5 else 'red'
            ax.plot(f1_thresholds[closest_value_idx], f1_scores[closest_value_idx], color=marker_color, marker='X', markersize=7)
        ax.set_xlim([-0.02, 1.02])    
        ax.set_ylim([-0.02, 1.02])
        ax.set_xlabel('threshold')
        ax.set_ylabel('F1')
        ax.legend(loc='lower center')
        ax.set_title(f'F1 Score') 

        # ROC
        ax = axs[1]    
        ax.plot(fpr, tpr, color=color, label=f'{type}, ROC AUC={roc_auc:.2f}')
        # setting crosses for some thresholds
        for threshold in (0.2, 0.4, 0.5, 0.6, 0.8):
            closest_value_idx = np.argmin(np.abs(roc_thresholds-threshold))
            marker_color = 'orange' if threshold != 0.5 else 'red'            
            ax.plot(fpr[closest_value_idx], tpr[closest_value_idx], color=marker_color, marker='X', markersize=7)
        ax.plot([0, 1], [0, 1], color='grey', linestyle='--')
        ax.set_xlim([-0.02, 1.02])    
        ax.set_ylim([-0.02, 1.02])
        ax.set_xlabel('FPR')
        ax.set_ylabel('TPR')
        ax.legend(loc='lower center')        
        ax.set_title(f'ROC Curve')
        
        # PRC
        ax = axs[2]
        ax.plot(recall, precision, color=color, label=f'{type}, AP={aps:.2f}')
        # setting crosses for some thresholds
        for threshold in (0.2, 0.4, 0.5, 0.6, 0.8):
            closest_value_idx = np.argmin(np.abs(pr_thresholds-threshold))
            marker_color = 'orange' if threshold != 0.5 else 'red'
            ax.plot(recall[closest_value_idx], precision[closest_value_idx], color=marker_color, marker='X', markersize=7)
        ax.set_xlim([-0.02, 1.02])    
        ax.set_ylim([-0.02, 1.02])
        ax.set_xlabel('recall')
        ax.set_ylabel('precision')
        ax.legend(loc='lower center')
        ax.set_title(f'PRC')        

        eval_stats[type]['Accuracy'] = metrics.accuracy_score(target, pred_target)
        eval_stats[type]['F1'] = metrics.f1_score(target, pred_target)
    
    df_eval_stats = pd.DataFrame(eval_stats)
    df_eval_stats = df_eval_stats.round(2)
    df_eval_stats = df_eval_stats.reindex(index=('Accuracy', 'F1', 'APS', 'ROC AUC'))
    
    print(df_eval_stats)
    
    return

In [ ]:
import re

def clear_text(text):
    
    text = text.lower()

    text = re.sub(r'[^a-z\s]', ' ', text)

    text = ' '.join(text.split())

    return text

In [ ]:
df_reviews['review_norm'] = df_reviews['review'].apply(clear_text) # <put your code here>

In [ ]:
df_reviews[['review', 'review_norm']].head()

In [ ]:
df_reviews_train = df_reviews.query('ds_part == "train"').copy()
df_reviews_test = df_reviews.query('ds_part == "test"').copy()

train_target = df_reviews_train['pos']
test_target = df_reviews_test['pos']

print(df_reviews_train.shape)
print(df_reviews_test.shape)

In [ ]:
from sklearn.dummy import DummyClassifier

In [ ]:
model_0 = DummyClassifier(
    strategy='most_frequent'
)

In [ ]:
model_0.fit(
    np.zeros((len(train_target),1)),
    train_target
)

In [ ]:
evaluate_model(
    model_0,
    np.zeros((len(train_target),1)),
    train_target,
    np.zeros((len(test_target),1)),
    test_target
)

In [ ]:
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from nltk.corpus import stopwords



In [ ]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000
)

In [ ]:
train_features_1 = tfidf.fit_transform(
    df_reviews_train['review_norm']
)

test_features_1 = tfidf.transform(
    df_reviews_test['review_norm']
)
print(train_features_1.shape)

In [ ]:


#Import:

from sklearn.linear_model import LogisticRegression

#Create:

model_1 = LogisticRegression(
    max_iter=1000
)



In [ ]:
#Train:

model_1.fit(
    train_features_1,
    train_target
)

In [ ]:

evaluate_model(
    model_1,
    train_features_1,
    train_target,
    test_features_1,
    test_target
)

In [ ]:
### Model 1 - NLTK, TF-IDF and LR

In [ ]:
#evaluate_model(model_1, train_features_1, train_target, test_features_1, test_target)

In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
print ('done')

In [ ]:
def text_preprocessing_3(text):
    
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    #tokens = [token.lemma_ for token in doc]
    
    return ' '.join(tokens)
print ('done')

In [ ]:
train_corpus_3 = df_reviews_train['review_norm'].progress_apply(text_preprocessing_3)
test_corpus_3 = df_reviews_test['review_norm'].progress_apply(text_preprocessing_3)

In [ ]:

tfidf_3 = TfidfVectorizer(
    max_features=10000
)


In [ ]:
train_features_3 = tfidf_3.fit_transform(
    train_corpus_3
)



In [ ]:
test_features_3 = tfidf_3.transform(
    test_corpus_3
)
print ('done')

In [ ]:

model_3 = LogisticRegression(
    max_iter=1000
)



In [ ]:
#Train:

model_3.fit(
    train_features_3,
    train_target
)

print ('done')

In [ ]:
evaluate_model(
    model_3,
    train_features_3,
    train_target,
    test_features_3,
    test_target
)

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
#Create Model
model_4 = LGBMClassifier(
    n_estimators=100
)

In [ ]:
#Convert to dense arrays:

train_features_4 = train_features_3
test_features_4 = test_features_3




In [ ]:
from lightgbm import LGBMClassifier


model_4 = LGBMClassifier(n_estimators=100, random_state=12345)

model_4.fit(train_features_4, train_target)

evaluate_model(model_4, train_features_4, train_target, test_features_4, test_target)

In [ ]:
#Train:

model_4.fit(
    train_features_4,
    train_target
)

In [ ]:

evaluate_model(
    model_4,
    train_features_4,
    train_target,
    test_features_4,
    test_target
)

In [ ]:
#Create a summary table.

from sklearn import metrics

def get_test_metrics(model, test_features, test_target):
    pred = model.predict(test_features)
    proba = model.predict_proba(test_features)[:, 1]
    return {
        'F1': metrics.f1_score(test_target, pred),
        'APS': metrics.average_precision_score(test_target, proba),
        'ROC AUC': metrics.roc_auc_score(test_target, proba),
    }

metrics_1 = get_test_metrics(model_1, test_features_1, test_target)
metrics_3 = get_test_metrics(model_3, test_features_3, test_target)
metrics_4 = get_test_metrics(model_4, test_features_4, test_target)

results = pd.DataFrame({
    'Model': ['TFIDF + LR', 'spaCy + TFIDF + LR', 'spaCy + TFIDF + LGBM'],
    'F1': [metrics_1['F1'], metrics_3['F1'], metrics_4['F1']],
    'APS': [metrics_1['APS'], metrics_3['APS'], metrics_4['APS']],
    'ROC AUC': [metrics_1['ROC AUC'], metrics_3['ROC AUC'], metrics_4['ROC AUC']]
})
print (results)

In [ ]:
print (results)

In [ ]:
#Sort values
results.sort_values(
    by='F1',
    ascending=False
)

In [ ]:
my_reviews = [
    "This movie was fantastic and I loved it.",
    "Terrible acting and boring plot.",
    "One of the best films I have ever seen.",
    "Waste of time and money."
]

In [ ]:
my_reviews_norm = [
    clear_text(x)
    for x in my_reviews
]

In [ ]:
my_features_1 = tfidf.transform(
    my_reviews_norm
)

#Predict:

model_1.predict(my_features_1)

In [ ]:
#Transform:

my_reviews_norm = [clear_text(review) for review in my_reviews]



my_features_1 = tfidf.transform(my_reviews_norm)

pred_1 = model_1.predict(my_features_1)

print(pred_1)

In [ ]:
my_reviews_spacy = [
    text_preprocessing_3(review)
    for review in my_reviews_norm
]


my_features_3 = tfidf_3.transform(my_reviews_spacy)

pred_3 = model_3.predict(my_features_3)

print(pred_3)

In [ ]:
my_features_4 = tfidf_3.transform(my_reviews_spacy)

pred_4 = model_4.predict(
    my_features_4.toarray()
)

print(pred_4)

In [ ]:

comparison = pd.DataFrame({
    'Review': my_reviews,
    'Model_1': pred_1,
    'Model_3': pred_3,
    'Model_4': pred_4
})

print (comparison)

In [ ]:
#import torch
#import transformers

In [ ]:
# tokenizer = transformers.BertTokenizer.from_pretrained('bert-base-uncased')
# config = transformers.BertConfig.from_pretrained('bert-base-uncased')
# model = transformers.BertModel.from_pretrained('bert-base-uncased')

In [ ]:
# def BERT_text_to_embeddings(texts, max_length=512, batch_size=100, force_device=None, disable_progress_bar=False):
    
#     ids_list = []
#     attention_mask_list = []

#     # text to padded ids of tokens along with their attention masks
    
#     # <put your code here to create ids_list and attention_mask_list>
    
#     if force_device is not None:
#         device = torch.device(force_device)
#     else:
#         device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
#     model.to(device)
#     if not disable_progress_bar:
#         print(f'Using the {device} device.')
    
#     # gettings embeddings in batches

#     embeddings = []

#     for i in tqdm(range(math.ceil(len(ids_list)/batch_size)), disable=disable_progress_bar):
            
#         ids_batch = torch.LongTensor(ids_list[batch_size*i:batch_size*(i+1)]).to(device)
#         # <put your code here to create attention_mask_batch
            
#         with torch.no_grad():            
#             model.eval()
#             batch_embeddings = model(input_ids=ids_batch, attention_mask=attention_mask_batch)   
#         embeddings.append(batch_embeddings[0][:,0,:].detach().cpu().numpy())
        
#     return np.concatenate(embeddings)

In [ ]:
# Attention! Running BERT for thousands of texts may take long run on CPU, at least several hours
# train_features_9 = BERT_text_to_embeddings(df_reviews_train['review_norm'], force_device='cuda')

In [ ]:
# 
# print(df_reviews_train['review_norm'].shape)
# print(train_features_9.shape)
# print(train_target.shape)


In [ ]:
##Project Not Completed

In [ ]:
# if you have got the embeddings, it's advisable to save them to have them ready if 
# np.savez_compressed('features_9.npz', train_features_9=train_features_9, test_features_9=test_features_9)

# and load...
# with np.load('features_9.npz') as data:
#     train_features_9 = data['train_features_9']
#     test_features_9 = data['test_features_9']

In [ ]:
# # feel free to completely remove these reviews and try your models on your own reviews, those below are just examples

# my_reviews = pd.DataFrame([
#     'I did not simply like it, not my kind of movie.',
#     'Well, I was bored and felt asleep in the middle of the movie.',
#     'I was really fascinated with the movie',    
#     'Even the actors looked really old and disinterested, and they got paid to be in the movie. What a soulless cash grab.',
#     'I didn\'t expect the reboot to be so good! Writers really cared about the source material',
#     'The movie had its upsides and downsides, but I feel like overall it\'s a decent flick. I could see myself going to see it again.',
#     'What a rotten attempt at a comedy. Not a single joke lands, everyone acts annoying and loud, even kids won\'t like this!',
#     'Launching on Netflix was a brave move & I really appreciate being able to binge on episode after episode, of this exciting intelligent new drama.'
# ], columns=['review'])

# my_reviews['review_norm'] = # <put here the same normalization logic as for the main dataset>

# my_reviews

In [ ]:
# texts = my_reviews['review_norm']

# my_reviews_pred_prob = model_2.predict_proba(tfidf_vectorizer_2.transform(texts))[:, 1]

# for i, review in enumerate(texts.str.slice(0, 100)):
#     print(f'{my_reviews_pred_prob[i]:.2f}:  {review}')

In [ ]:
# texts = my_reviews['review_norm']

# my_reviews_pred_prob = model_3.predict_proba(tfidf_vectorizer_3.transform(texts.apply(lambda x: text_preprocessing_3(x))))[:, 1]

# for i, review in enumerate(texts.str.slice(0, 100)):
#     print(f'{my_reviews_pred_prob[i]:.2f}:  {review}')

In [ ]:
# texts = my_reviews['review_norm']

# tfidf_vectorizer_4 = tfidf_vectorizer_3
# my_reviews_pred_prob = model_4.predict_proba(tfidf_vectorizer_4.transform(texts.apply(lambda x: text_preprocessing_3(x))))[:, 1]

# for i, review in enumerate(texts.str.slice(0, 100)):
#     print(f'{my_reviews_pred_prob[i]:.2f}:  {review}')